In [1]:
# N-1 — Environment. Afterwards: Runtime → Restart session, then run N-2 onward.
!apt-get -qq install -y git-lfs > /dev/null
import subprocess
from pathlib import Path
ROOT = Path("/content/Project-phase-2")
if not ROOT.exists():
    subprocess.run(["git", "clone", "https://github.com/CMU-18734-17731-2026-Fall/Project-phase-2.git", str(ROOT)], check=True)
subprocess.run(["git", "lfs", "install"], check=True)
subprocess.run(["git", "lfs", "pull"], cwd=ROOT, check=True)
%pip install -q "transformers==4.56.2" "peft==0.13.2" "accelerate==1.10.1" "datasets==4.1.1"
print("Done. Restart the session now.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 137.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 124.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requ

In [2]:
# N-2 — Imports and constants. Nothing inherited from earlier runtimes.
import os, sys, gc, json, random, shutil, hashlib, subprocess
os.environ.update(USE_TF="0", USE_FLAX="0", TOKENIZERS_PARALLELISM="false")
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn.functional as F
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, roc_curve
import transformers, peft
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
assert transformers.__version__ == "4.56.2" and peft.__version__ == "0.13.2"
assert torch.cuda.is_available(), "GPU runtime required"

from google.colab import drive
drive.mount("/content/drive")

ROOT = Path("/content/Project-phase-2")
DRIVE = Path("/content/drive/MyDrive/MIA_phase2")
OLD = DRIVE / "cache"
MODEL_NAME, MAX_LENGTH = "gpt2_3_lora32_adamw_b8_lr2", 512
PHASES = ("train", "val", "final")                    # "train" = the dev target's evaluation texts
PAIRS = [("inA", "inB"), ("p2A", "p2B"), ("p3A", "p3B"), ("p4A", "p4B")]
NEW_PAIRS = [("p5A", "p5B"), ("p6A", "p6B")]
NEW_PAIR_NO = {("p5A", "p5B"): 5, ("p6A", "p6B"): 6}
REF_NAMES = [n for p in PAIRS for n in p]
KAPPA = 8
device, model_dtype = torch.device("cuda"), torch.float16
env = os.environ.copy()
assert ROOT.exists() and DRIVE.exists()
print("OK")

Mounted at /content/drive
OK


In [3]:
# N-3 — Texts in original order, labels, tokenizer, membership, global evaluation groups.
def read_json(p):
    with Path(p).open("r", encoding="utf-8") as f:
        return json.load(f)

TEXTS = {ph: read_json(ROOT / "data" / ph / "test.json") for ph in PHASES}
LABELS = np.asarray(read_json(ROOT / "data" / "train" / "test_label.json"), dtype=np.int64)
assert all(len(TEXTS[ph]) == 2000 for ph in PHASES) and LABELS.sum() == 1000
DEV_ORDER_SHA = hashlib.sha256("\x1e".join(TEXTS["train"]).encode()).hexdigest()

ADAPTER_DIR = ROOT / "models" / "train" / MODEL_NAME
adapter_config = read_json(ADAPTER_DIR / "adapter_config.json")
BASE_MODEL_NAME = adapter_config["base_model_name_or_path"]
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
SHADOW_TRAIN_SIZE = len(read_json(ROOT / "data" / "train" / "train_finetune.json"))
assert SHADOW_TRAIN_SIZE == 10000

norm = lambda t: " ".join(t.split())
def prefix_keys(texts):
    keys = []
    for s in range(0, len(texts), 256):
        for ids in tokenizer(texts[s:s + 256], add_special_tokens=False, truncation=True,
                             max_length=MAX_LENGTH, padding=False)["input_ids"]:
            keys.append(hashlib.sha256(np.asarray(ids, dtype="<i4").tobytes()).hexdigest())
    return keys
EVAL_NORM = {ph: [norm(t) for t in TEXTS[ph]] for ph in PHASES}
EVAL_PREF = {ph: prefix_keys(TEXTS[ph]) for ph in PHASES}

def membership(train_texts):
    ns = {norm(t) for t in train_texts}; ps = set(prefix_keys(list(train_texts)))
    return {ph: np.array([(a in ns) or (b in ps) for a, b in zip(EVAL_NORM[ph], EVAL_PREF[ph])]) for ph in PHASES}
def shadow_texts(n):
    return [r if isinstance(r, str) else r["text"] for r in read_json(DRIVE / "data" / f"shadow_{n}" / "train_finetune.json")]

MEM = {n: membership(shadow_texts(n)) for n in REF_NAMES}
for a, b in PAIRS[1:]:
    for ph in PHASES:
        assert (MEM[a][ph] ^ MEM[b][ph]).all(), (a, b, ph)
assert int((MEM["inA"]["train"] & MEM["inB"]["train"]).sum()) == 7

# Global groups over dev+val+final (same item order and keys as the original construction).
EVAL_ITEMS = [(ph, i) for ph in PHASES for i in range(2000)]
parent = list(range(len(EVAL_ITEMS)))
def _find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
first = {}
for k, (ph, i) in enumerate(EVAL_ITEMS):
    for key in (("n", EVAL_NORM[ph][i]), ("p", EVAL_PREF[ph][i])):
        if key in first: parent[_find(k)] = _find(first[key])
        else: first[key] = k
GROUPS = {}
for k in range(len(EVAL_ITEMS)):
    GROUPS.setdefault(_find(k), []).append(k)
EVAL_NORM_SET = {EVAL_NORM[ph][i] for ph, i in EVAL_ITEMS}
EVAL_PREF_SET = {EVAL_PREF[ph][i] for ph, i in EVAL_ITEMS}

def assign_sides(pair_no):
    rng = np.random.default_rng(7000 + pair_no)
    side_items, side_n = ([], []), [0, 0]
    for g in rng.permutation(sorted(GROUPS)):
        g = int(g)
        s = int(rng.integers(2)) if side_n[0] == side_n[1] else int(side_n[1] < side_n[0])
        side_items[s].extend(GROUPS[g]); side_n[s] += len(GROUPS[g])
    return side_items, side_n

# Reproduction check: the assignment rule must regenerate pairs 2–4 exactly from their saved datasets.
for k in (2, 3, 4):
    side_items, _ = assign_sides(k)
    flag = np.zeros(len(EVAL_ITEMS), bool); flag[side_items[1]] = True
    got = np.concatenate([MEM[f"p{k}B"][ph] for ph in PHASES])
    assert np.array_equal(flag, got), f"group assignment does not reproduce pair {k}"
print(f"{len(EVAL_ITEMS)} eval items in {len(GROUPS)} groups; assignment rule reproduces pairs 2–4.")

6000 eval items in 5981 groups; assignment rule reproduces pairs 2–4.


In [4]:
# N-4 — Cached losses, alignment checks, frozen LR4 reproduction, bootstrap indices.
def split_npz(path, key="losses"):
    z = np.load(path)
    return [a.astype(np.float64) for a in np.split(z[key], np.cumsum(z["lengths"])[:-1])]

TGT = {"train": split_npz(OLD / "train_base_target_losses.npz", "target_losses"),
       "val":   split_npz(OLD / "val_ref_train_losses.npz"),
       "final": split_npz(OLD / "final_train_tgt_losses.npz")}
REF = {n: {ph: split_npz(OLD / f"{ph}_shadow_{n}_losses.npz") for ph in PHASES} for n in REF_NAMES}
for ph in PHASES:
    exp_len = np.array([len(ids) - 1 for ids in tokenizer(TEXTS[ph], truncation=True, max_length=MAX_LENGTH)["input_ids"]])
    assert np.array_equal(exp_len, [a.size for a in TGT[ph]]), ph
    for n in REF_NAMES:
        assert all(r.shape == t.shape for r, t in zip(REF[n][ph], TGT[ph])), (n, ph)

fixed = [split_npz(OLD / "train_shadow_1001_losses.npz", "shadow_losses"), split_npz(OLD / "train_shadow_1002_losses.npz"),
         split_npz(OLD / "train_ref_val_losses.npz"), split_npz(OLD / "train_ref_final_losses.npz")]
FIX4 = np.stack([np.array([a.mean() for a in f]) for f in fixed])   # shadow1001, shadow1002, val tgt, final tgt
DEV_MT = np.array([a.mean() for a in TGT["train"]])
DEV_L = np.array([a.size for a in TGT["train"]], dtype=np.float64)
mean_loss = lambda n: np.array([a.mean() for a in REF[n]["train"]])

def tpr_strict(y, s):
    fpr, tpr, _ = roc_curve(y, s)
    return float(np.max(tpr[fpr < 0.01]))
def strict_fast(s, y):
    neg, pos = s[y == 0], s[y == 1]
    return np.mean(pos > np.sort(neg)[::-1][int(np.ceil(0.01 * neg.size)) - 1])

def lr_frozen(mt, L, P, Pin, kappa=KAPPA):
    """Frozen LR4: unequal-variance Gaussian LR on √T·mean loss, κ-shrinkage to pooled IN/OUT variances, ×(1+1/n)."""
    sq = np.sqrt(L); U, u = P * sq, mt * sq
    fin = np.isfinite(P); m_in, m_out = Pin & fin, ~Pin & fin
    n_in, n_out = m_in.sum(0), m_out.sum(0)
    assert (n_in > 0).all() and (n_out > 0).all()
    Uz = np.where(fin, U, 0.0)
    mu_in = np.where(m_in, Uz, 0.0).sum(0) / n_in
    mu_out = np.where(m_out, Uz, 0.0).sum(0) / n_out
    rss_in = np.where(m_in, (Uz - mu_in) ** 2, 0.0).sum(0)
    rss_out = np.where(m_out, (Uz - mu_out) ** 2, 0.0).sum(0)
    df_in, df_out = np.maximum(n_in - 1, 0), np.maximum(n_out - 1, 0)
    pool_out = rss_out.sum() / df_out.sum()
    pool_in = rss_in.sum() / df_in.sum() if df_in.sum() > 0 else pool_out
    s_in = (rss_in + kappa * pool_in) / (df_in + kappa) * (1 + 1 / n_in)
    s_out = (rss_out + kappa * pool_out) / (df_out + kappa) * (1 + 1 / n_out)
    A = (-(u - mu_in) ** 2 / (2 * s_in) - 0.5 * np.log(s_in)) - (-(u - mu_out) ** 2 / (2 * s_out) - 0.5 * np.log(s_out))
    assert np.isfinite(A).all()
    return A

S4 = np.load(OLD / "staged4_dev_scores_4pairs.npz")
LR4_DEV = S4["LR (all OUT, κ=8 fixed)"].astype(np.float64)
P_lr4 = np.vstack([np.stack([mean_loss(n) for n in REF_NAMES]), FIX4])
PIN_lr4 = np.vstack([np.stack([MEM[n]["train"] for n in REF_NAMES]), np.zeros(FIX4.shape, bool)])
assert np.allclose(lr_frozen(DEV_MT, DEV_L, P_lr4, PIN_lr4), LR4_DEV, rtol=0, atol=1e-6), "LR4 not reproduced"
assert np.isclose(tpr_strict(LABELS, LR4_DEV), read_json(DRIVE / "frozen_LR4_k8" / "settings.json")["dev"]["LR"])

# Same STAGED-5 mock numbers as before (pairs 1–2 refs + shadows 1001/1002), as a second reproduction check.
R0 = ["inA", "inB", "p2A", "p2B"]
P0 = np.vstack([np.stack([mean_loss(n) for n in R0]), FIX4[:2]])
PIN0 = np.vstack([np.stack([MEM[n]["train"] for n in R0]), np.zeros((2, 2000), bool)])
for t, exp in (("p3A", 0.8483), ("p3B", 0.8084), ("p4A", 0.7450), ("p4B", 0.6586)):
    got = tpr_strict(MEM[t]["train"].astype(int), lr_frozen(mean_loss(t), DEV_L, P0, PIN0))
    assert abs(got - exp) < 6e-4, (t, got, exp)

rng = np.random.default_rng(20261017)                         # same dev bootstrap as earlier notebooks
pos_i, neg_i = np.flatnonzero(LABELS == 1), np.flatnonzero(LABELS == 0)
boot = [np.concatenate([rng.choice(pos_i, pos_i.size), rng.choice(neg_i, neg_i.size)]) for _ in range(2000)]
print(f"Frozen LR4 reproduced: AUC={roc_auc_score(LABELS, LR4_DEV):.4f} | strict TPR={tpr_strict(LABELS, LR4_DEV):.4f}; "
      "STAGED-5 mock values reproduced.")

Frozen LR4 reproduced: AUC=0.9744 | strict TPR=0.7120; STAGED-5 mock values reproduced.


In [5]:
# P-0 — Status of p5/p6 and helpers.
for sub in ("models", "data", "logs"):
    (DRIVE / sub).mkdir(exist_ok=True)
def status(n):
    return {"data_on_drive": (DRIVE / "data" / f"shadow_{n}" / "train_finetune.json").exists(),
            "adapter_on_drive": (DRIVE / "models" / f"shadow_{n}" / "adapter_model.safetensors").exists(),
            "losses_cached": all((OLD / f"{ph}_shadow_{n}_losses.npz").exists() for ph in PHASES)}
print(pd.DataFrame({n: status(n) for p in NEW_PAIRS for n in p}).T)

def train_cmd(data_dir, out_dir, seed):
    args = [sys.executable, str(ROOT / "ft_llm" / "train_shadow_model.py"),
            "--data-dir", str(data_dir), "--outdir", str(out_dir), "--model-name", BASE_MODEL_NAME,
            "--epochs", "3", "--batch-size", "2", "--gradient-accumulation-steps", "4",
            "--block-size", "512", "--lr", "2e-4", "--weight-decay", "0.01", "--seed", str(seed),
            "--lora-r", str(adapter_config["r"]), "--lora-alpha", str(adapter_config["lora_alpha"]),
            "--lora-dropout", str(adapter_config["lora_dropout"]),
            "--target-modules", ",".join(adapter_config["target_modules"]),
            "--precision", "auto", "--gradient-checkpointing"]
    return " ".join(f"'{a}'" for a in args)

@torch.inference_mode()
def collect_losses(model, texts, batch_size=2):
    """Same as the caching for all earlier shadows: fp16 forward, CE on fp32 logits, batch 2, right padding."""
    out = []
    for s in tqdm(range(0, len(texts), batch_size), leave=False):
        b = tokenizer(texts[s:s + batch_size], return_tensors="pt", padding=True, truncation=True, max_length=MAX_LENGTH)
        b = {k: v.to(device) for k, v in b.items()}
        tgt, mask = b["input_ids"][:, 1:], b["attention_mask"][:, 1:].bool()
        logits = model(**b, use_cache=False).logits[:, :-1, :]
        loss = F.cross_entropy(logits.float().reshape(-1, logits.shape[-1]), tgt.reshape(-1),
                               reduction="none").reshape(tgt.shape)
        for row, m in zip(loss, mask):
            out.append(row[m].float().cpu().numpy())
    return out

     data_on_drive  adapter_on_drive  losses_cached
p5A          False             False          False
p5B          False             False          False
p6A          False             False          False
p6B          False             False          False


In [6]:
# P-1 — Datasets for p5/p6: same global assignment rule as pairs 2–4, fresh WikiText auxiliary data per model.
FRESH_GEN_BASE = 46000                                   # seeds 46050/46051, 46060/46061 (p4 used 44001/44002)
for (a, b) in NEW_PAIRS:
    k = NEW_PAIR_NO[(a, b)]
    have = [status(n)["data_on_drive"] for n in (a, b)]
    if all(have):
        print(f"pair {k}: data already on Drive, skipping build"); continue
    assert not any(have), f"pair {k}: only one dataset exists; inspect before rebuilding (nothing overwritten)"

    side_items, side_n = assign_sides(k)
    new_mem, record = {}, {"pair_no": k, "names": [a, b], "group_seed": 7000 + k, "side_sizes": side_n,
                           "aux_source": "FRESH independent WikiText-103 draws per model, eval-overlap excluded",
                           "models": {}}
    for s, name in enumerate((a, b)):
        gen_seed = FRESH_GEN_BASE + 10 * k + s
        cand_dir = ROOT / "data" / f"fresh_candidates_{name}"
        r = subprocess.run([sys.executable, str(ROOT / "data" / "prepare_finetuning_data.py"), "--seed", str(gen_seed),
                            "--num-samples", "12000", "--min-tokens", "25", "--model-name", BASE_MODEL_NAME,
                            "--output-dir", str(cand_dir)], env=env, capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stderr[-2000:]); raise RuntimeError(f"candidate generation failed for {name}")
        cands = list(dict.fromkeys(x if isinstance(x, str) else x["text"] for x in read_json(cand_dir / "train_finetune.json")))
        pk = prefix_keys(cands)
        clean = [t for t, p in zip(cands, pk) if norm(t) not in EVAL_NORM_SET and p not in EVAL_PREF_SET]
        ev = list(dict.fromkeys(TEXTS[EVAL_ITEMS[j][0]][EVAL_ITEMS[j][1]] for j in side_items[s]))
        n_aux = SHADOW_TRAIN_SIZE - len(ev)
        assert len(clean) >= n_aux, f"{name}: need {n_aux}, have {len(clean)}"
        aux = random.Random(8000 + 10 * k + s).sample(clean, n_aux)
        tt = ev + aux
        random.Random(9000 + 10 * k + s).shuffle(tt)
        assert len(tt) == SHADOW_TRAIN_SIZE
        d = ROOT / "data" / f"shadow_{name}"; d.mkdir(parents=True, exist_ok=True)
        with (d / "train_finetune.json").open("w", encoding="utf-8") as f:
            json.dump([{"text": t} for t in tt], f, ensure_ascii=False)
        shutil.copytree(d, DRIVE / "data" / f"shadow_{name}", dirs_exist_ok=True)      # immediate backup
        new_mem[name] = membership(tt)
        record["models"][name] = {"fresh_generation_seed": gen_seed, "aux_sampling_seed": 8000 + 10 * k + s,
                                  "shuffle_seed": 9000 + 10 * k + s, "train_seed": 5000 + 10 * k + s,
                                  "candidates": len(cands), "after_eval_exclusion": len(clean), "eval_in": len(ev)}
    for ph in PHASES:
        assert (new_mem[a][ph] ^ new_mem[b][ph]).all(), f"pair {k} {ph}: an eval item is not IN exactly one model"
    flag = np.zeros(len(EVAL_ITEMS), int); flag[side_items[1]] = 1
    record["side_by_phase"] = {ph: flag[j * 2000:(j + 1) * 2000].tolist() for j, ph in enumerate(PHASES)}
    # Auxiliary overlap with existing models (shared data only; recorded, not used by the attack).
    aux_sets = {n: {norm(t) for t in shadow_texts(n)} - EVAL_NORM_SET for n in (a, b)}
    record["aux_overlap"] = {n: {m: len(aux_sets[n] & ({norm(t) for t in shadow_texts(m)} - EVAL_NORM_SET))
                                 for m in REF_NAMES + [x for x in (a, b) if x != n]} for n in (a, b)}
    with open(DRIVE / "data" / f"pair{k}_assignment.json", "w") as f:
        json.dump(record, f, indent=1)
    print(f"pair {k}: built and backed up | eval IN:", {n: record['models'][n]['eval_in'] for n in (a, b)})
    print("  aux overlap:", record["aux_overlap"])
print(pd.DataFrame({n: status(n) for p in NEW_PAIRS for n in p}).T)

pair 5: built and backed up | eval IN: {'p5A': 2988, 'p5B': 2993}
  aux overlap: {'p5A': {'inA': 67, 'inB': 58, 'p2A': 67, 'p2B': 80, 'p3A': 63, 'p3B': 75, 'p4A': 61, 'p4B': 75, 'p5B': 70}, 'p5B': {'inA': 76, 'inB': 60, 'p2A': 56, 'p2B': 70, 'p3A': 70, 'p3B': 72, 'p4A': 75, 'p4B': 88, 'p5A': 70}}
pair 6: built and backed up | eval IN: {'p6A': 2987, 'p6B': 2994}
  aux overlap: {'p6A': {'inA': 58, 'inB': 64, 'p2A': 58, 'p2B': 56, 'p3A': 63, 'p3B': 60, 'p4A': 64, 'p4B': 58, 'p6B': 75}, 'p6B': {'inA': 72, 'inB': 68, 'p2A': 69, 'p2B': 83, 'p3A': 76, 'p3B': 74, 'p4A': 58, 'p4B': 73, 'p6A': 75}}
     data_on_drive  adapter_on_drive  losses_cached
p5A           True             False          False
p5B           True             False          False
p6A           True             False          False
p6B           True             False          False


In [7]:
# P-1b — Train every p5/p6 model without an adapter on Drive, sequentially, with per-model backup.
todo = [n for p in NEW_PAIRS for n in p if not status(n)["adapter_on_drive"]]
assert all(status(n)["data_on_drive"] for n in todo), "run P-1 first"
steps = []
for n in todo:
    k, s = int(n[1]), int(n.endswith("B"))
    out_dir, log = ROOT / "models" / f"shadow_{n}", ROOT / f"train_shadow_{n}.log"
    steps.append(f"{train_cmd(DRIVE / 'data' / f'shadow_{n}', out_dir, 5000 + 10 * k + s)} > '{log}' 2>&1")
    steps.append(f"cp -r '{out_dir}' '{DRIVE}/models/' && cp '{log}' '{DRIVE}/logs/'")
if todo:
    train_proc = subprocess.Popen(" && ".join(steps), shell=True, cwd=ROOT, env=env,
                                  stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f"Training {todo} sequentially (PID {train_proc.pid}); ~25 min per model.")
    print(f"Progress: !tail -c 400 {ROOT}/train_shadow_{todo[0]}.log   |   finished? train_proc.poll()")
else:
    print("All p5/p6 adapters already on Drive.")

Training ['p5A', 'p5B', 'p6A', 'p6B'] sequentially (PID 8075); ~25 min per model.
Progress: !tail -c 400 /content/Project-phase-2/train_shadow_p5A.log   |   finished? train_proc.poll()


In [23]:
# P-2 — Score each p5/p6 model whose adapter is on Drive and whose losses are not yet cached.
for n in [n for p in NEW_PAIRS for n in p]:
    st = status(n)
    if not st["adapter_on_drive"] or st["losses_cached"]:
        continue
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, torch_dtype=model_dtype)
    m = PeftModel.from_pretrained(base, DRIVE / "models" / f"shadow_{n}")
    m.config.pad_token_id = tokenizer.pad_token_id; m.to(device).eval()
    for ph in PHASES:
        ls = collect_losses(m, TEXTS[ph])
        assert len(ls) == 2000 and all(x.size == t.size for x, t in zip(ls, TGT[ph]))
        np.savez_compressed(OLD / f"{ph}_shadow_{n}_losses.npz",
                            lengths=np.asarray([x.size for x in ls]), losses=np.concatenate(ls))
    m.to("cpu"); del m, base; gc.collect(); torch.cuda.empty_cache()
    print(f"{n}: losses cached on Drive")

for n in [n for p in NEW_PAIRS for n in p]:
    if status(n)["losses_cached"] and n not in REF:
        REF[n] = {ph: split_npz(OLD / f"{ph}_shadow_{n}_losses.npz") for ph in PHASES}
        MEM[n] = membership(shadow_texts(n))
for (a, b) in NEW_PAIRS:
    if a in REF and b in REF:
        for ph in PHASES:
            assert (MEM[a][ph] ^ MEM[b][ph]).all(), (a, b, ph)
print("Loaded:", [n for p in NEW_PAIRS for n in p if n in REF])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

p5A: losses cached on Drive


...

p6A: losses cached on Drive


  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

p6B: losses cached on Drive
Loaded: ['p5A', 'p5B', 'p6A', 'p6B']


In [22]:
print(pd.DataFrame({n: status(n) for p in NEW_PAIRS for n in p}).T)
print("training still running" if 'train_proc' in globals() and train_proc.poll() is None else "no training process running")

     data_on_drive  adapter_on_drive  losses_cached
p5A           True              True          False
p5B           True              True          False
p6A           True              True          False
p6B           True              True          False
no training process running


In [24]:
# P-3 — Frozen LR at 4/5/6 pairs vs LR4 on dev; fixed fresh-mock comparison (targets p4A/p4B).
LR56_CACHE = DRIVE / "cache" / "lr_pairs56"; LR56_CACHE.mkdir(parents=True, exist_ok=True)
complete = [p for p in NEW_PAIRS if all(n in REF for n in p)]
assert complete, "p5 is not fully scored yet"
stage_pairs = {4: list(PAIRS)}
for j in range(len(complete)):
    stage_pairs[5 + j] = list(PAIRS) + complete[:j + 1]

dev_scores = {}
for st, prs in stage_pairs.items():
    names = [n for p in prs for n in p]
    P = np.vstack([np.stack([mean_loss(n) for n in names]), FIX4])
    Pin = np.vstack([np.stack([MEM[n]["train"] for n in names]), np.zeros(FIX4.shape, bool)])
    dev_scores[st] = lr_frozen(DEV_MT, DEV_L, P, Pin)
assert np.allclose(dev_scores[4], LR4_DEV, rtol=0, atol=1e-6)

bt = {st: np.array([strict_fast(v[ix], LABELS[ix]) for ix in boot]) for st, v in dev_scores.items()}
rows = []
for st, v in dev_scores.items():
    d = bt[st] - bt[4]; lo, hi = np.percentile(d, [2.5, 97.5])
    rows.append({"stage": f"{st} pairs" + (" (= LR4)" if st == 4 else ""), "refs": 2 * st + 4,
                 "dev AUC": roc_auc_score(LABELS, v), "strict TPR": tpr_strict(LABELS, v),
                 "Δ TPR vs LR4": tpr_strict(LABELS, v) - tpr_strict(LABELS, LR4_DEV),
                 "95% CI": f"[{lo:+.3f}, {hi:+.3f}]", "P(Δ>0)": float(np.mean(d > 0))})
dev_tab = pd.DataFrame(rows)
print("Dev — frozen LR; references = paired shadows of the stage + shadow1001, shadow1002, val tgt, final tgt:")
print(dev_tab.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

mock_refs = {"pairs 1–3": ["inA", "inB", "p2A", "p2B", "p3A", "p3B"]}
if len(complete) >= 1: mock_refs["pairs 1–3 + p5"] = mock_refs["pairs 1–3"] + list(complete[0])
if len(complete) >= 2: mock_refs["pairs 1–3 + p5 + p6"] = mock_refs["pairs 1–3 + p5"] + list(complete[1])
mrows, mock_scores = [], {}
for tgt_name in ("p4A", "p4B"):
    y = MEM[tgt_name]["train"].astype(int); mt = mean_loss(tgt_name)
    pos_t, neg_t = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    rng_t = np.random.default_rng(31 + (tgt_name == "p4B"))
    boot_t = [np.concatenate([rng_t.choice(pos_t, pos_t.size), rng_t.choice(neg_t, neg_t.size)]) for _ in range(2000)]
    base_bt = base_tpr = None
    for label, names in mock_refs.items():
        assert "p4A" not in names and "p4B" not in names
        P = np.vstack([np.stack([mean_loss(n) for n in names]), FIX4[:2]])
        Pin = np.vstack([np.stack([MEM[n]["train"] for n in names]), np.zeros((2, 2000), bool)])
        v = lr_frozen(mt, DEV_L, P, Pin)
        mock_scores[f"{tgt_name}|{label}"] = v
        btv = np.array([strict_fast(v[ix], y[ix]) for ix in boot_t])
        if base_bt is None: base_bt, base_tpr = btv, tpr_strict(y, v)
        d = btv - base_bt; lo, hi = np.percentile(d, [2.5, 97.5])
        mrows.append({"target": tgt_name, "references": label, "refs": len(names) + 2,
                      "TPR": tpr_strict(y, v), "AUC": roc_auc_score(y, v),
                      "Δ TPR vs pairs 1–3": tpr_strict(y, v) - base_tpr, "95% CI": f"[{lo:+.3f}, {hi:+.3f}]"})
mock_tab = pd.DataFrame(mrows)
print("\nFresh mock — targets p4A/p4B, pair 4 excluded, + shadows 1001/1002, κ=8, same dev texts at every stage:")
print(mock_tab.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

np.savez(LR56_CACHE / f"lr_stage_dev_scores_upto{max(stage_pairs)}.npz", labels=LABELS,
         dev_order_sha=np.array(DEV_ORDER_SHA), **{f"stage_{st}_pairs": v for st, v in dev_scores.items()})
np.savez(LR56_CACHE / f"lr_stage_fresh_mock_scores_upto{max(stage_pairs)}.npz",
         **{k.replace(" ", "_").replace("+", "plus").replace("–", "-").replace("|", "__"): v for k, v in mock_scores.items()})
dev_tab.to_csv(LR56_CACHE / f"lr_stage_dev_upto{max(stage_pairs)}.csv", index=False)
mock_tab.to_csv(LR56_CACHE / f"lr_stage_fresh_mock_upto{max(stage_pairs)}.csv", index=False)
with open(LR56_CACHE / f"lr_stage_notes_upto{max(stage_pairs)}.json", "w") as f:
    json.dump({"formula": "frozen LR4 (unequal-variance LR on sqrt(T)*mean loss, kappa=8, pooled shrinkage, 1+1/n, finite masks)",
               "dev_references": "stage's paired shadows by actual membership + shadow1001, shadow1002, val target, final target (OUT)",
               "stages": {st: [list(p) for p in prs] for st, prs in stage_pairs.items()},
               "fresh_mock": "targets p4A/p4B; refs pairs 1-3 (+p5, +p6) + shadows 1001/1002; pair 4 excluded",
               "variants_or_sweeps": False, "validation_submission": False}, f, indent=2)
print("\nSaved to", LR56_CACHE)

Dev — frozen LR; references = paired shadows of the stage + shadow1001, shadow1002, val tgt, final tgt:
          stage  refs  dev AUC  strict TPR  Δ TPR vs LR4           95% CI  P(Δ>0)
4 pairs (= LR4)    12   0.9744      0.7120        0.0000 [+0.000, +0.000]  0.0000
        5 pairs    14   0.9756      0.7260        0.0140 [-0.022, +0.046]  0.5580
        6 pairs    16   0.9771      0.7340        0.0220 [-0.049, +0.050]  0.7965

Fresh mock — targets p4A/p4B, pair 4 excluded, + shadows 1001/1002, κ=8, same dev texts at every stage:
target          references  refs    TPR    AUC  Δ TPR vs pairs 1–3           95% CI
   p4A           pairs 1–3     8 0.7311 0.9770              0.0000 [+0.000, +0.000]
   p4A      pairs 1–3 + p5    10 0.7779 0.9821              0.0468 [-0.019, +0.093]
   p4A pairs 1–3 + p5 + p6    12 0.7759 0.9827              0.0448 [-0.035, +0.088]
   p4B           pairs 1–3     8 0.7048 0.9712              0.0000 [+0.000, +0.000]
   p4B      pairs 1–3 + p5    10 0.7550 0.9

In [25]:
# V-6 — Single candidate: frozen LR formula (κ=8), 6 pairs + LR4's fixed OUT refs. Builds val + final ZIPs only.
import zipfile, io
FZ4 = DRIVE / "frozen_LR4_k8"
FZ6 = DRIVE / "frozen_LR6_k8"; FZ6.mkdir(exist_ok=True)
PAIRS6 = list(PAIRS) + [("p5A", "p5B"), ("p6A", "p6B")]
NAMES4, NAMES6 = [n for p in PAIRS for n in p], [n for p in PAIRS6 for n in p]
assert all(n in REF and n in MEM for n in NAMES6)
tm = lambda arrs: np.array([a.mean() for a in arrs])
txt = lambda r: r if isinstance(r, str) else r["text"]

def find_data(n):
    for p in (DRIVE / "data" / f"shadow_{n}", DRIVE / f"data_shadow_{n}", ROOT / "data" / f"shadow_{n}"):
        if (p / "train_finetune.json").exists():
            return p / "train_finetune.json"
    raise FileNotFoundError(f"training data for shadow_{n}")
mem_s = {n: membership([txt(r) for r in read_json(find_data(n))]) for n in ("1001", "1002")}
assert not mem_s["1001"]["val"].any() and not mem_s["1002"]["val"].any()
train_ft = {norm(txt(r)) for r in read_json(ROOT / "data" / "train" / "train_finetune.json")}
in_tf = {ph: np.array([norm(t) in train_ft for t in TEXTS[ph]]) for ph in ("val", "final")}
assert int(in_tf["val"].sum()) == 39
print("Final texts masked | shadow1001:", int(mem_s["1001"]["final"].sum()), "| shadow1002:",
      int(mem_s["1002"]["final"].sum()), "| train target:", int(in_tf["final"].sum()))

# Targets and fixed OUT references, exactly as in LR4.
VT = split_npz(OLD / "val_shadow_1001_results.npz", "target_losses")
FL = {k: split_npz(OLD / f"final_{k}_losses.npz") for k in ("target", "shadow1", "shadow2", "train_tgt", "val_tgt")}
VR = {"s1": split_npz(OLD / "val_shadow_1001_results.npz", "shadow_losses"), "s2": split_npz(OLD / "val_shadow_1002_losses.npz"),
      "fin": split_npz(OLD / "val_ref_final_losses.npz"), "trn": split_npz(OLD / "val_ref_train_losses.npz")}
for arrs, tgt in [(v, VT) for v in VR.values()] + [(v, FL["target"]) for v in FL.values()] + \
                 [(REF[n]["val"], VT) for n in NAMES6] + [(REF[n]["final"], FL["target"]) for n in NAMES6]:
    assert len(arrs) == 2000 and all(a.shape == b.shape for a, b in zip(arrs, tgt))
TARGET = {"val": (tm(VT), np.array([a.size for a in VT], float)),
          "final": (tm(FL["target"]), np.array([a.size for a in FL["target"]], float))}
FIXED = {"val": np.stack([tm(VR["s1"]), tm(VR["s2"]), tm(VR["fin"]), np.where(in_tf["val"], np.nan, tm(VR["trn"]))]),
         "final": np.stack([np.where(mem_s["1001"]["final"], np.nan, tm(FL["shadow1"])),
                            np.where(mem_s["1002"]["final"], np.nan, tm(FL["shadow2"])),
                            np.where(in_tf["final"], np.nan, tm(FL["train_tgt"])), tm(FL["val_tgt"])])}

def score(ph, names):
    mt, L = TARGET[ph]; fx = FIXED[ph]
    P = np.vstack([np.stack([tm(REF[n][ph]) for n in names]), fx])
    Pin = np.vstack([np.stack([MEM[n][ph] for n in names]), np.zeros(fx.shape, bool)])
    s = lr_frozen(mt, L, P, Pin)
    fin = np.isfinite(P)
    return s, (Pin & fin).sum(0), (~Pin & fin).sum(0)

# Reproduction check: 4 pairs must reproduce the frozen LR4 val and final scores.
for ph, key in (("val", "val_LR"), ("final", "final_LR")):
    saved = np.load(FZ4 / f"{ph}_scores.npz")[key]
    s4 = score(ph, NAMES4)[0]
    d = np.abs(s4 - saved).max()
    print(f"{ph}: |rebuilt LR4 − saved LR4| max = {d:.2e}")
    assert d < 1e-4, f"{ph}: LR4 not reproduced; stop"

# Dev 6-pair must match P-3.
assert np.allclose(dev_scores[6], lr_frozen(DEV_MT, DEV_L,
    np.vstack([np.stack([mean_loss(n) for n in NAMES6]), FIX4]),
    np.vstack([np.stack([MEM[n]["train"] for n in NAMES6]), np.zeros(FIX4.shape, bool)])), rtol=0, atol=1e-9)

out = {}
for ph in ("val", "final"):
    s6, n_in, n_out = score(ph, NAMES6)
    assert s6.shape == (2000,) and np.isfinite(s6).all()
    s4 = score(ph, NAMES4)[0]
    out[ph] = s6
    print(f"{ph}: IN counts {dict(zip(*np.unique(n_in, return_counts=True)))} | OUT counts "
          f"{dict(zip(*np.unique(n_out, return_counts=True)))} | corr(LR6, LR4) = {np.corrcoef(s6, s4)[0, 1]:.4f} | "
          f"mean {s6.mean():.3f} sd {s6.std():.3f}")
    pred = ROOT / "data" / ph / "prediction_LR6.csv"
    zp = ROOT / f"project_phase2_{ph}_submission_LR6_k8.zip"
    np.savetxt(pred, s6, fmt="%.17g")
    for cmd in ([sys.executable, str(ROOT / "scripts" / "make_submission.py"), "--phase", ph,
                 "--prediction", str(pred), "--output", str(zp)],
                [sys.executable, str(ROOT / "scripts" / "validate_submission.py"), str(zp), "--phase", ph]):
        r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
        print(r.stdout.strip(), r.stderr.strip()); assert r.returncode == 0
    with zipfile.ZipFile(zp) as zf:
        assert np.allclose(np.loadtxt(io.BytesIO(zf.read(f"{ph}/prediction.csv"))), s6, rtol=0, atol=1e-9)
    shutil.copy(zp, FZ6); shutil.copy(zp, DRIVE)
    np.savez(FZ6 / f"{ph}_scores.npz", **{f"{ph}_LR6": s6, "n_in": n_in, "n_out": n_out})

json.dump({"method": "frozen LR4 formula (unequal-variance Gaussian LR on sqrt(T)·mean loss, κ=8, pooled shrinkage, "
                     "1+1/n, finite masks) with 6 IN/OUT pairs",
           "pairs": [list(p) for p in PAIRS6], "fresh_aux_pairs": ["p4", "p5", "p6"],
           "val_fixed_out_refs": ["shadow1001", "shadow1002", "final target", "train target (39 masked)"],
           "final_fixed_out_refs": ["shadow1001 (masked)", "shadow1002 (masked)", "train target (masked)", "val target"],
           "selection": "inspected 5- and 6-pair stages; 6 pairs highest on dev TPR/AUC and fresh-mock mean TPR/AUC",
           "dev": {"LR4": 0.712, "5 pairs": 0.726, "6 pairs": 0.734},
           "fresh_mock_mean_TPR": {"pairs 1-3": 0.718, "+p5": 0.766, "+p5+p6": 0.784},
           "validation_submission": "built, not uploaded"}, open(FZ6 / "settings.json", "w"), indent=2)
print("LR6 val and final ZIPs built and saved to", FZ6, "| LR4 artifacts untouched.")

Final texts masked | shadow1001: 26 | shadow1002: 26 | train target: 31
val: |rebuilt LR4 − saved LR4| max = 5.33e-15
final: |rebuilt LR4 − saved LR4| max = 3.55e-15
val: IN counts {np.int64(6): np.int64(1993), np.int64(7): np.int64(7)} | OUT counts {np.int64(8): np.int64(3), np.int64(9): np.int64(40), np.int64(10): np.int64(1957)} | corr(LR6, LR4) = 0.9869 | mean 1.221 sd 7.501
Created /content/Project-phase-2/project_phase2_val_submission_LR6_k8.zip with 2000 scores 
Submission validation: PASS (2000 scores)
Score range: [-20.0461, 47.175] 
final: IN counts {np.int64(6): np.int64(1977), np.int64(7): np.int64(23)} | OUT counts {np.int64(8): np.int64(21), np.int64(9): np.int64(64), np.int64(10): np.int64(1915)} | corr(LR6, LR4) = 0.9817 | mean 1.139 sd 7.738
Created /content/Project-phase-2/project_phase2_final_submission_LR6_k8.zip with 2000 scores 
Submission validation: PASS (2000 scores)
Score range: [-21.4992, 46.9397] 
LR6 val and final ZIPs built and saved to /content/drive/MyDr

In [26]:
# D-1 — LR4 vs LR6 on validation: reproduction, integrity, rank diagnostics. No labels, no submission.
from scipy.stats import spearmanr, rankdata
DIAG = DRIVE / "cache" / "lr_pairs56" / "diag_val"; DIAG.mkdir(parents=True, exist_ok=True)
def zip_scores(path, ph):
    with zipfile.ZipFile(path) as zf:
        return np.loadtxt(io.BytesIO(zf.read(f"{ph}/prediction.csv")))
rank_of = lambda s: rankdata(-s, method="ordinal").astype(int)        # 1 = most member-like

# ---------- 1. Reproduction against the SUBMITTED scores ----------
s4, nin4, nout4 = score("val", NAMES4)
s6, nin6, nout6 = score("val", NAMES6)
sub4 = zip_scores(FZ4 / "project_phase2_val_submission_LR4_k8.zip", "val")
sav4 = np.load(FZ4 / "val_scores.npz")["val_LR"]
sub6 = zip_scores(FZ6 / "project_phase2_val_submission_LR6_k8.zip", "val")
rep = []
for name, a, b in (("rebuilt LR4 vs submitted LR4 ZIP", s4, sub4), ("rebuilt LR4 vs saved LR4 npz", s4, sav4),
                   ("submitted LR4 ZIP vs saved npz", sub4, sav4), ("rebuilt LR6 vs submitted LR6 ZIP", s6, sub6)):
    rep.append({"check": name, "max |Δ|": float(np.abs(a - b).max()), "Spearman": spearmanr(a, b)[0],
                "max |rank Δ|": int(np.abs(rank_of(a) - rank_of(b)).max()),
                "top-750 identical": bool(np.array_equal(np.sort(np.argsort(-a)[:750]), np.sort(np.argsort(-b)[:750])))})
rep = pd.DataFrame(rep)
print("1. Reproduction:"); print(rep.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
REPRO_OK = bool(rep.iloc[0]["max |Δ|"] < 1e-4 and rep.iloc[0]["max |rank Δ|"] == 0 and rep.iloc[3]["max |Δ|"] < 1e-9)
print("LR4 reproduces the submitted scores:", REPRO_OK)

# ---------- 2a. Text order ----------
g = subprocess.run(["git", "diff", "--quiet", "HEAD", "--", "data/val/test.json"], cwd=ROOT)
tok_len = np.array([len(ids) - 1 for ids in tokenizer(TEXTS["val"], truncation=True, max_length=MAX_LENGTH)["input_ids"]])
caches = {"val target": VT, "train target": VR["trn"], "final target": VR["fin"], "shadow1001": VR["s1"],
          "shadow1002": VR["s2"]} | {n: REF[n]["val"] for n in NAMES6}
bad_len = [k for k, v in caches.items() if not np.array_equal([a.size for a in v], tok_len)]
print(f"\n2a. data/val/test.json unchanged vs repo: {g.returncode == 0} | sha256 of val texts: "
      f"{hashlib.sha256(chr(30).join(TEXTS['val']).encode()).hexdigest()[:16]} | caches misaligned: {bad_len or 'none'}")

# ---------- 2b. Adapter identities: sha256 + recompute 16 val texts vs cache ----------
def adapter_dir(name):
    for p in (DRIVE / "models" / name, DRIVE / name, ROOT / "models" / name):
        if (p / "adapter_model.safetensors").exists():
            return p
    raise FileNotFoundError(name)
ADAPTERS = {"val target": ROOT / "models" / "val" / MODEL_NAME, "train target": ADAPTER_DIR,
            "final target": ROOT / "models" / "final" / MODEL_NAME,
            "shadow1001": adapter_dir("shadow_1001"), "shadow1002": adapter_dir("shadow_1002")} | \
           {n: adapter_dir(f"shadow_{n}") for n in NAMES6}
ids_rows = []
for k, path in ADAPTERS.items():
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, torch_dtype=model_dtype)
    m = PeftModel.from_pretrained(base, path); m.config.pad_token_id = tokenizer.pad_token_id; m.to(device).eval()
    fresh = collect_losses(m, TEXTS["val"][:16])
    d = max(float(np.abs(f.astype(np.float64) - c).max()) for f, c in zip(fresh, caches[k][:16]))
    ids_rows.append({"model": k, "adapter sha256": hashlib.sha256((Path(path) / "adapter_model.safetensors").read_bytes()).hexdigest()[:16],
                     "path": str(path), "max |fresh − cached| (16 texts)": d})
    m.to("cpu"); del m, base; gc.collect(); torch.cuda.empty_cache()
ids_tab = pd.DataFrame(ids_rows)
print("\n2b. Adapter identity (fresh recompute vs cached val losses):")
print(ids_tab.drop(columns="path").to_string(index=False, float_format=lambda x: f"{x:.2e}"))

# ---------- 2c. p5/p6 membership on validation ----------
mrows = []
for (a, b), k in ((("p5A", "p5B"), 5), (("p6A", "p6B"), 6)):
    rec = json.load(open(DRIVE / "data" / f"pair{k}_assignment.json"))
    side_b = np.array(rec["side_by_phase"]["val"], bool)
    for n, expect in ((a, ~side_b), (b, side_b)):
        fresh_mem = membership([txt(r) for r in read_json(DRIVE / "data" / f"shadow_{n}" / "train_finetune.json")])["val"]
        mrows.append({"model": n, "val IN": int(fresh_mem.sum()), "matches loaded MEM": bool(np.array_equal(fresh_mem, MEM[n]["val"])),
                      "matches assignment record": bool(np.array_equal(fresh_mem, expect))})
    assert (MEM[a]["val"] ^ MEM[b]["val"]).all()
print("\n2c. p5/p6 validation membership (XOR holds for both pairs):"); print(pd.DataFrame(mrows).to_string(index=False))

# ---------- 2d. The 39 train-target exclusions ----------
tt = [txt(r) for r in read_json(ROOT / "data" / "train" / "train_finetune.json")]
mem_tt = membership(tt)["val"]                                       # text OR scored-prefix rule
nan_mask = np.isnan(FIXED["val"][3])
print(f"\n2d. Train-target exclusions | exact-text rule: {int(in_tf['val'].sum())} | text-or-prefix rule: {int(mem_tt.sum())} "
      f"| NaN mask = exact-text set: {bool(np.array_equal(nan_mask, in_tf['val']))} "
      f"| prefix-only matches not masked: {np.flatnonzero(mem_tt & ~in_tf['val']).tolist()}")

# ---------- 3. Rank comparison (only if LR4 reproduces) ----------
def lr_parts(mt, L, P, Pin, kappa=KAPPA if "KAPPA" in globals() else 8):
    sq = np.sqrt(L); U, u = P * sq, mt * sq
    fin = np.isfinite(P); m_in, m_out = Pin & fin, ~Pin & fin
    n_in, n_out = m_in.sum(0), m_out.sum(0)
    Uz = np.where(fin, U, 0.0)
    mu_in = np.where(m_in, Uz, 0).sum(0) / n_in; mu_out = np.where(m_out, Uz, 0).sum(0) / n_out
    rss_in = np.where(m_in, (Uz - mu_in) ** 2, 0).sum(0); rss_out = np.where(m_out, (Uz - mu_out) ** 2, 0).sum(0)
    df_in, df_out = np.maximum(n_in - 1, 0), np.maximum(n_out - 1, 0)
    pool_out = rss_out.sum() / df_out.sum(); pool_in = rss_in.sum() / df_in.sum() if df_in.sum() > 0 else pool_out
    s_in = (rss_in + kappa * pool_in) / (df_in + kappa) * (1 + 1 / n_in)
    s_out = (rss_out + kappa * pool_out) / (df_out + kappa) * (1 + 1 / n_out)
    lr = (-(u - mu_in) ** 2 / (2 * s_in) - 0.5 * np.log(s_in)) - (-(u - mu_out) ** 2 / (2 * s_out) - 0.5 * np.log(s_out))
    return dict(lr=lr, u=u, mu_in=mu_in, mu_out=mu_out, s_in=s_in, s_out=s_out, n_in=n_in, n_out=n_out,
                pool_in=pool_in, pool_out=pool_out)
def build(names):
    mt, L = TARGET["val"]; fx = FIXED["val"]
    P = np.vstack([np.stack([tm(REF[n]["val"]) for n in names]), fx])
    Pin = np.vstack([np.stack([MEM[n]["val"] for n in names]), np.zeros(fx.shape, bool)])
    return lr_parts(mt, L, P, Pin)

if not REPRO_OK:
    print("\nLR4 did not reproduce the submitted scores: rank comparison skipped. Send this output.")
else:
    A, B = build(NAMES4), build(NAMES6)
    assert np.allclose(A["lr"], s4, atol=1e-9) and np.allclose(B["lr"], s6, atol=1e-9)
    r4, r6 = rank_of(s4), rank_of(s6)
    top = lambda s, k: set(np.argsort(-s)[:k])
    print(f"\n3. Spearman(LR4, LR6) = {spearmanr(s4, s6)[0]:.4f} | Pearson = {np.corrcoef(s4, s6)[0, 1]:.4f}")
    print("Top-k overlap:", {k: f"{len(top(s4, k) & top(s6, k))}/{k}" for k in (100, 250, 500)},
          f"| k=750 (near the strict-TPR threshold): {len(top(s4, 750) & top(s6, 750))}/750")
    print(f"Pooled variances  LR4: in {A['pool_in']:.4f} out {A['pool_out']:.4f} | "
          f"LR6: in {B['pool_in']:.4f} out {B['pool_out']:.4f}")
    print(f"Score scale  LR4 mean {s4.mean():.2f} sd {s4.std():.2f} | LR6 mean {s6.mean():.2f} sd {s6.std():.2f}")

    sqL = np.sqrt(TARGET["val"][1])
    new = ["p5A", "p5B", "p6A", "p6B"]
    def describe(idx, why):
        rows = []
        for i in idx:
            r = {"why": why, "text": int(i), "rank LR4": int(r4[i]), "rank LR6": int(r6[i]), "u": A["u"][i]}
            for tag, X in (("4", A), ("6", B)):
                r |= {f"μin{tag}": X["mu_in"][i], f"μout{tag}": X["mu_out"][i], f"s_in{tag}": X["s_in"][i],
                      f"s_out{tag}": X["s_out"][i], f"n{tag} in/out": f"{X['n_in'][i]}/{X['n_out'][i]}"}
            r["new refs (U, IN?)"] = ", ".join(f"{n}:{tm([REF[n]['val'][i]])[0] * sqL[i]:.2f}{'I' if MEM[n]['val'][i] else 'O'}"
                                               for n in new)
            rows.append(r)
        return rows
    drank = r4 - r6                                                     # > 0: moved up under LR6
    into = [i for i in np.argsort(-drank) if i in top(s6, 750) and i not in top(s4, 750)][:10]
    out_ = [i for i in np.argsort(drank) if i in top(s4, 750) and i not in top(s6, 750)][:10]
    big = np.argsort(-np.abs(drank))[:10]
    mv = pd.DataFrame(describe(into, "entered top-750 under LR6") + describe(out_, "left top-750 under LR6")
                      + describe(big, "largest |rank change| overall"))
    pd.set_option("display.width", 320); pd.set_option("display.max_columns", 40)
    print("\nLargest movers (no labels):"); print(mv.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
    print(f"\nTexts entering/leaving the top-750: {len(top(s6, 750) - top(s4, 750))} each")
    mv.to_csv(DIAG / "movers.csv", index=False)
rep.to_csv(DIAG / "reproduction.csv", index=False); ids_tab.to_csv(DIAG / "adapter_identity.csv", index=False)
pd.DataFrame(mrows).to_csv(DIAG / "p56_val_membership.csv", index=False)
print("\nSaved to", DIAG)

1. Reproduction:
                           check  max |Δ|  Spearman  max |rank Δ|  top-750 identical
rebuilt LR4 vs submitted LR4 ZIP 5.33e-15         1             0               True
    rebuilt LR4 vs saved LR4 npz 5.33e-15         1             0               True
  submitted LR4 ZIP vs saved npz        0         1             0               True
rebuilt LR6 vs submitted LR6 ZIP        0         1             0               True
LR4 reproduces the submitted scores: True

2a. data/val/test.json unchanged vs repo: True | sha256 of val texts: a6cc9f8ee5e34364 | caches misaligned: none


  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

...

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

[... log truncated ...]
  0.048      6/10 p5A:22.65I, p5B:22.71O, p6A:22.44O, p6B:22.65I
       left top-750 under LR6   145       671       761 23.002 22.904 23.536  0.068   0.040       4/8 22.917 23.486  0.060   0.043      6/10 p5A:23.41O, p5B:23.12I, p6A:22.77I, p6B:23.16O
       left top-750 under LR6  1610       709       789 40.449 40.054 41.008  0.082   0.038       4/8 40.047 40.971  0.066   0.037      6/10 p5A:40.80O, p5B:39.99I, p6A:40.84O, p6B:40.08I
       left top-750 under LR6  1796       727       800 17.696 17.699 18.194  0.075   0.043       4/8 17.710 18.159  0.063   0.048      6/10 p5A:17.86I, p5B:17.77O, p6A:18.26O, p6B:17.60I
       left top-750 under LR6   876       748       819 34.854 34.718 35.398  0.070   0.057       4/8 34.741 35.369  0.063   0.069      6/10 p5A:34.97I, p5B:34.89O, p6A:35.62O, p6B:34.61I
       left top-750 under LR6   308       733       792 25.212 25.256 25.650  0.058   0.034       4/8 25.261 25.612  0.050   0.034      6/10 p5A:25.43O, p5B:25

In [27]:
# BAG-1 — Reference-subset bagging of the frozen LR (κ=8): all 4-pair subsets; mean and median aggregation.
from itertools import combinations
BAG = DRIVE / "cache" / "lr_pairs56" / "bagging"; BAG.mkdir(parents=True, exist_ok=True)
PAIRS6 = list(PAIRS) + [("p5A", "p5B"), ("p6A", "p6B")]
assert all(n in REF and n in MEM for p in PAIRS6 for n in p)

def lr_pool(pairs, mt, fixed=FIX4):
    names = [n for p in pairs for n in p]
    P = np.vstack([np.stack([mean_loss(n) for n in names]), fixed])
    Pin = np.vstack([np.stack([MEM[n]["train"] for n in names]), np.zeros(fixed.shape, bool)])
    return lr_frozen(mt, DEV_L, P, Pin)

def paired_ci(a, b, y, bidx):
    d = np.array([strict_fast(a[ix], y[ix]) - strict_fast(b[ix], y[ix]) for ix in bidx])
    lo, hi = np.percentile(d, [2.5, 97.5])
    return tpr_strict(y, a) - tpr_strict(y, b), f"[{lo:+.3f}, {hi:+.3f}]", float(np.mean(d > 0))

# ---------- Dev: 15 subsets of 4 of the 6 pairs ----------
SUBS = list(combinations(range(6), 4)); assert len(SUBS) == 15
SUB_DEV = np.stack([lr_pool([PAIRS6[j] for j in s], DEV_MT) for s in SUBS])
assert np.allclose(SUB_DEV[SUBS.index((0, 1, 2, 3))], LR4_DEV, rtol=0, atol=1e-6), "subset {1,2,3,4} must equal LR4"
LR6_DEV = lr_pool(PAIRS6, DEV_MT)
assert np.allclose(LR6_DEV, dev_scores[6], rtol=0, atol=1e-9), "pooled 6 pairs must equal LR6 (P-3)"
DEVM = {"LR4": LR4_DEV, "LR6 (pooled 6 pairs)": LR6_DEV,
        "bag-mean (15 × 4 pairs)": SUB_DEV.mean(0), "bag-median (15 × 4 pairs)": np.median(SUB_DEV, 0)}
rows = []
for k, v in DEVM.items():
    r = {"method": k, "dev AUC": roc_auc_score(LABELS, v), "strict TPR": tpr_strict(LABELS, v)}
    for base in ("LR4", "LR6 (pooled 6 pairs)"):
        r[f"Δ vs {base.split()[0]}"], r[f"CI vs {base.split()[0]}"], r[f"P(Δ>0) vs {base.split()[0]}"] = \
            paired_ci(v, DEVM[base], LABELS, boot)
    rows.append(r)
dev_tab = pd.DataFrame(rows)
pd.set_option("display.width", 260)
print("Dev (real target; fixed OUT refs = shadow1001, shadow1002, val tgt, final tgt; κ=8):")
print(dev_tab.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
sub_tpr = np.array([tpr_strict(LABELS, s) for s in SUB_DEV])
print(f"[diagnostic only] individual 4-pair subsets on dev: TPR min {sub_tpr.min():.3f} | median {np.median(sub_tpr):.3f} "
      f"| max {sub_tpr.max():.3f} | LR4's subset {sub_tpr[SUBS.index((0, 1, 2, 3))]:.3f}")
print(f"Score correlations: corr(bag-mean, LR6) = {np.corrcoef(DEVM['bag-mean (15 × 4 pairs)'], LR6_DEV)[0, 1]:.4f}, "
      f"corr(bag-mean, LR4) = {np.corrcoef(DEVM['bag-mean (15 × 4 pairs)'], LR4_DEV)[0, 1]:.4f}")

# ---------- Fresh mocks: drop the target's pair first, then all 4-pair subsets of the remaining 5 ----------
REM = [p for p in PAIRS6 if p != ("p4A", "p4B")]          # pairs 1, 2, 3, 5, 6
MSUBS = list(combinations(range(5), 4)); assert len(MSUBS) == 5
mrows, mock_scores = [], {}
for tgt in ("p4A", "p4B"):
    y, mt = MEM[tgt]["train"].astype(int), mean_loss(tgt)
    pos_t, neg_t = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    rng_t = np.random.default_rng(31 + (tgt == "p4B"))
    bt_t = [np.concatenate([rng_t.choice(pos_t, pos_t.size), rng_t.choice(neg_t, neg_t.size)]) for _ in range(2000)]
    subs = np.stack([lr_pool([REM[j] for j in s], mt) for s in MSUBS])
    M = {"pairs 1–3 (LR4's remaining pairs)": lr_pool(REM[:3], mt), "pooled 5 pairs (LR6 analog)": lr_pool(REM, mt),
         "bag-mean (5 × 4 pairs)": subs.mean(0), "bag-median (5 × 4 pairs)": np.median(subs, 0)}
    for k, v in M.items():
        mock_scores[f"{tgt}|{k}"] = v
        r = {"target": tgt, "method": k, "TPR": tpr_strict(y, v), "AUC": roc_auc_score(y, v)}
        for base, short in (("pairs 1–3 (LR4's remaining pairs)", "pairs1–3"), ("pooled 5 pairs (LR6 analog)", "pooled")):
            r[f"Δ vs {short}"], r[f"CI vs {short}"], _ = paired_ci(v, M[base], y, bt_t)
        mrows.append(r)
    st = np.array([tpr_strict(y, s) for s in subs])
    print(f"[diagnostic only] {tgt}: individual 4-pair subsets TPR min {st.min():.3f} | max {st.max():.3f}")
mock_tab = pd.DataFrame(mrows)
print("\nFresh mocks (pair 4 excluded; same 4 fixed OUT refs for every method; κ=8; dev texts):")
print(mock_tab.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nMock mean over p4A/p4B:")
print(mock_tab.groupby("method")[["TPR", "AUC", "Δ vs pairs1–3", "Δ vs pooled"]].mean().to_string(float_format=lambda x: f"{x:.4f}"))

np.savez(BAG / "dev_scores.npz", labels=LABELS, dev_order_sha=np.array(DEV_ORDER_SHA), subsets=np.array(SUBS), sub_scores=SUB_DEV,
         **{k.split(" ")[0].replace("-", "_"): v for k, v in DEVM.items()})
np.savez(BAG / "mock_scores.npz", **{k.replace(" ", "_").replace("|", "__").replace("–", "-").replace("(", "").replace(")", "")
                                     .replace("×", "x").replace("'", ""): v for k, v in mock_scores.items()})
dev_tab.to_csv(BAG / "dev.csv", index=False); mock_tab.to_csv(BAG / "fresh_mock.csv", index=False)
json.dump({"subsets_dev": "all 15 four-pair subsets of 6 pairs", "subsets_mock": "all 5 four-pair subsets of the 5 non-target pairs",
           "fixed_out_refs": "shadow1001, shadow1002, val target, final target (all methods, dev and mock)",
           "formula": "frozen LR4 per subset (κ=8, pooled shrinkage within subset, 1+1/n, finite masks)",
           "aggregations": ["mean", "median"], "no_weighting_or_subset_selection": True,
           "validation_used": False, "submission": False}, open(BAG / "notes.json", "w"), indent=2)
print("\nSaved to", BAG)

Dev (real target; fixed OUT refs = shadow1001, shadow1002, val tgt, final tgt; κ=8):
                   method  dev AUC  strict TPR  Δ vs LR4        CI vs LR4  P(Δ>0) vs LR4  Δ vs LR6        CI vs LR6  P(Δ>0) vs LR6
                      LR4   0.9744      0.7120    0.0000 [+0.000, +0.000]         0.0000   -0.0220 [-0.050, +0.049]         0.1905
     LR6 (pooled 6 pairs)   0.9771      0.7340    0.0220 [-0.049, +0.050]         0.7965    0.0000 [+0.000, +0.000]         0.0000
  bag-mean (15 × 4 pairs)   0.9772      0.7460    0.0340 [-0.032, +0.047]         0.6965    0.0120 [-0.070, +0.033]         0.6695
bag-median (15 × 4 pairs)   0.9772      0.7400    0.0280 [-0.048, +0.041]         0.6540    0.0060 [-0.085, +0.025]         0.5740
[diagnostic only] individual 4-pair subsets on dev: TPR min 0.685 | median 0.730 | max 0.763 | LR4's subset 0.712
Score correlations: corr(bag-mean, LR6) = 0.9974, corr(bag-mean, LR4) = 0.9886
[diagnostic only] p4A: individual 4-pair subsets TPR min 0.785 | ma

In [33]:
# ENS-T — Frozen E-3 ensembles (LR, LR+B_ext, LR+B_ext+RMIA_a) on p5A, p5B, p6A, p6B. No refitting.
from scipy.special import logsumexp
ENS_CACHE = DRIVE / "cache" / "ensemble"
OUTT = ENS_CACHE / "transfer_p56"; OUTT.mkdir(exist_ok=True)
notes = json.load(open(ENS_CACHE / "ens_notes.json"))
CO = notes["coefficients_E3"]
FROZEN_NAMES = ["inA", "inB", "p2A", "p2B", "p3A", "p3B", "p4A", "p4B"]
assert notes["models"] == FROZEN_NAMES
TARGETS = ["p5A", "p5B", "p6A", "p6B"]
assert not set(TARGETS) & set(FROZEN_NAMES) and all(t in REF and t in MEM for t in TARGETS)
SQ = np.sqrt(DEV_L)
ML4 = {n: mean_loss(n) for n in FROZEN_NAMES}

def log_trim_mean_exp(L, prop=0.2):
    K = L.shape[0]; c = int(prop * K)
    S = np.sort(L, axis=0)[c:K - c]
    return logsumexp(S, axis=0) - np.log(S.shape[0])

def ens_features(mt):
    """Exactly E-0's real-target features: 8 frozen paired shadows (by membership) + the 4 fixed OUT refs."""
    P = np.vstack([np.stack([ML4[n] for n in FROZEN_NAMES]), FIX4])
    Pin = np.vstack([np.stack([MEM[n]["train"] for n in FROZEN_NAMES]), np.zeros(FIX4.shape, bool)])
    out = ~Pin & np.isfinite(P); n_out = out.sum(0); assert (n_out > 0).all()
    F = {"LR": lr_frozen(mt, DEV_L, P, Pin),
         "B_ext": (np.where(out, P, 0.0).sum(0) / n_out - mt) * SQ,
         "RMIA_a": -mt - log_trim_mean_exp(-np.stack([ML4[n] for n in FROZEN_NAMES]))}
    assert all(np.isfinite(v).all() and v.shape == (2000,) for v in F.values())
    return F, {"LR refs": P.shape[0], "IN min–max": f"{(Pin & np.isfinite(P)).sum(0).min()}–{(Pin & np.isfinite(P)).sum(0).max()}",
               "OUT min–max": f"{n_out.min()}–{n_out.max()}", "RMIA refs": len(FROZEN_NAMES)}

def apply_frozen(fs, F):
    c = CO[fs]; cols = list(c["standardized_w"])                      # saved feature order
    X = np.column_stack([F[k] for k in cols])
    mu = np.array([c["scaler_mean"][k] for k in cols]); sd = np.array([c["scaler_scale"][k] for k in cols])
    w = np.array([c["standardized_w"][k] for k in cols])
    return c["intercept"] + ((X - mu) / sd) @ w

# 1. The saved pipeline must reproduce its real-dev scores.
REAL_F, real_info = ens_features(DEV_MT)
assert np.allclose(REAL_F["LR"], LR4_DEV, rtol=0, atol=1e-6)
saved = np.load(ENS_CACHE / "ens_dev_scores.npz")
for fs in CO:
    d = np.abs(apply_frozen(fs, REAL_F) - saved[fs.replace("+", "_plus_")]).max()
    print(f"real dev reproduction | {fs}: max |Δ| = {d:.2e}")
    assert d < 1e-8, f"{fs}: frozen pipeline does not reproduce saved dev scores"

# 2. Transfer to p5/p6 (dev texts; shadow membership as labels).
rows, info_rows, scores = [], [{"target": "real dev", **real_info}], {}
draws_d = {}
for t in TARGETS:
    y = MEM[t]["train"].astype(int)
    F, info = ens_features(mean_loss(t)); info_rows.append({"target": t, **info})
    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    rng = np.random.default_rng(500 + TARGETS.index(t))
    bidx = [np.concatenate([rng.choice(pos, pos.size), rng.choice(neg, neg.size)]) for _ in range(2000)]
    S = {fs: apply_frozen(fs, F) for fs in CO}
    base_bt = np.array([strict_fast(S["LR"][ix], y[ix]) for ix in bidx])
    for fs, v in S.items():
        scores[f"{t}__{fs.replace('+', '_plus_')}"] = v
        bt = np.array([strict_fast(v[ix], y[ix]) for ix in bidx]); d = bt - base_bt
        draws_d[(t, fs)] = d
        lo, hi = np.percentile(d, [2.5, 97.5])
        rows.append({"target": t, "set": fs, "n members": int(y.sum()), "TPR": tpr_strict(y, v), "AUC": roc_auc_score(y, v),
                     "Δ TPR vs LR": tpr_strict(y, v) - tpr_strict(y, S["LR"]), "95% CI": f"[{lo:+.3f}, {hi:+.3f}]",
                     "P(Δ>0)": float(np.mean(d > 0))})
tab = pd.DataFrame(rows)
print("\nReference counts (p5/p6 match the real dev target):"); print(pd.DataFrame(info_rows).to_string(index=False))
pd.set_option("display.width", 220)
print("\nFrozen ensembles on p5/p6 (never used for fitting, C, or feature choice):")
print(tab.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
summ = []
for fs in CO:
    sub = tab[tab.set == fs]
    dm = np.mean([draws_d[(t, fs)] for t in TARGETS], axis=0)       # independent per-target resamples, averaged
    lo, hi = np.percentile(dm, [2.5, 97.5])
    summ.append({"set": fs, "mean TPR": sub.TPR.mean(), "mean AUC": sub.AUC.mean(), "mean Δ TPR vs LR": sub["Δ TPR vs LR"].mean(),
                 "95% CI (mean Δ)": f"[{lo:+.3f}, {hi:+.3f}]", "targets with Δ>0": int((sub["Δ TPR vs LR"] > 0).sum())})
summ = pd.DataFrame(summ)
print("\nMean across the four targets:"); print(summ.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

tab.to_csv(OUTT / "per_target.csv", index=False); summ.to_csv(OUTT / "summary.csv", index=False)
np.savez(OUTT / "scores.npz", dev_order_sha=np.array(DEV_ORDER_SHA), **scores)
json.dump({"classifiers": "E-3 frozen (coefficients, intercepts, scalers from ens_notes.json); no refitting",
           "features": "8 frozen paired shadows + shadow1001, shadow1002, val tgt, final tgt; RMIA on 8 paired, trim 0.2",
           "targets": TARGETS, "labels": "shadow training membership on dev texts", "validation_submission": False},
          open(OUTT / "notes.json", "w"), indent=2)
print("Saved to", OUTT)

real dev reproduction | LR: max |Δ| = 0.00e+00
real dev reproduction | LR+B_ext: max |Δ| = 0.00e+00
real dev reproduction | LR+B_ext+RMIA_a: max |Δ| = 0.00e+00

Reference counts (p5/p6 match the real dev target):
  target  LR refs IN min–max OUT min–max  RMIA refs
real dev       12        4–5         7–8          8
     p5A       12        4–5         7–8          8
     p5B       12        4–5         7–8          8
     p6A       12        4–5         7–8          8
     p6B       12        4–5         7–8          8

Frozen ensembles on p5/p6 (never used for fitting, C, or feature choice):
target             set  n members    TPR    AUC  Δ TPR vs LR           95% CI  P(Δ>0)
   p5A              LR        979 0.7467 0.9794       0.0000 [+0.000, +0.000]  0.0000
   p5A        LR+B_ext        979 0.7150 0.9786      -0.0317 [-0.079, +0.019]  0.1020
   p5A LR+B_ext+RMIA_a        979 0.7549 0.9784       0.0082 [-0.054, +0.045]  0.5530
   p5B              LR       1021 0.7287 0.9778       0.

In [34]:
# TAIL-1 — Tail-focused linear combination (pairwise logistic vs top-5% non-members), held-out-pair × text folds.
MEMD = {n: MEM[n]["train"].copy() for n in ["inA", "inB", "p2A", "p2B", "p3A", "p3B", "p4A", "p4B"]}
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
TAIL = DRIVE / "cache" / "ensemble" / "tail_rank"; TAIL.mkdir(exist_ok=True)
FROZEN_PAIRS = [("inA", "inB"), ("p2A", "p2B"), ("p3A", "p3B"), ("p4A", "p4B")]
PARTNER4 = {a: b for a, b in FROZEN_PAIRS} | {b: a for a, b in FROZEN_PAIRS}
FRESH = {"p4A", "p4B"}
COLS = ["LR", "B_ext", "RMIA_a"]
LAMBDAS, FRAC, ROUNDS, C_ORD = (0.01, 0.1, 1.0, 10.0), 0.05, 10, 0.01          # declared in advance

def feats(mt, excl=()):
    names = [n for n in FROZEN_NAMES if n not in set(excl)]
    P = np.vstack([np.stack([ML4[n] for n in names]), FIX4])
    Pin = np.vstack([np.stack([MEM[n]["train"] for n in names]), np.zeros(FIX4.shape, bool)])
    out = ~Pin & np.isfinite(P); n_out = out.sum(0); assert (n_out > 0).all()
    F = {"LR": lr_frozen(mt, DEV_L, P, Pin), "B_ext": (np.where(out, P, 0.0).sum(0) / n_out - mt) * SQ,
         "RMIA_a": -mt - log_trim_mean_exp(-np.stack([ML4[n] for n in names]))}
    assert all(np.isfinite(v).all() for v in F.values())
    return F, set(names)
_F0, _ = feats(DEV_MT)
assert all(np.allclose(_F0[k], REAL_F[k], rtol=0, atol=1e-12) for k in COLS)      # same as E-0 / ENS-T

def tail_fit(Zs, ys, lam, w0, frac=FRAC, rounds=ROUNDS):
    """Equal-weight mean over targets of mean pairwise logistic loss (members vs current top-frac non-members)."""
    w, prev = w0.astype(float).copy(), None
    for r in range(rounds):
        hard = []
        for Z, y in zip(Zs, ys):
            neg = np.flatnonzero(y == 0); k = max(1, int(np.ceil(frac * neg.size)))
            hard.append(np.sort(neg[np.argsort(-(Z[neg] @ w))[:k]]))
        key = tuple(map(tuple, hard))
        if key == prev:
            break
        prev = key
        def obj(v):
            L, G = 0.0, np.zeros_like(v)
            for Z, y, h in zip(Zs, ys, hard):
                Pz, Nz = Z[y == 1], Z[h]
                D = (Pz @ v)[:, None] - (Nz @ v)[None, :]
                L += np.logaddexp(0.0, -D).mean()
                g = -expit(-D) / D.size
                G += g.sum(1) @ Pz - g.sum(0) @ Nz
            L /= len(Zs); G /= len(Zs)
            return L + lam * ((v - w0) ** 2).sum(), G + 2 * lam * (v - w0)
        w = minimize(obj, w, jac=True, method="L-BFGS-B").x
    return w, r + 1

def alpha0(Zs, ys):
    a, _ = tail_fit([Z[:, :1] for Z in Zs], ys, 0.0, np.array([1.0]))
    assert a[0] > 0
    return a[0]

dev_root = np.array([_find(k) for k in range(2000)])
perm = np.random.default_rng(20261018).permutation(np.unique(dev_root))       # same text folds as E-2
fold_of = {g: j % 2 for j, g in enumerate(perm)}
TFOLD = np.array([fold_of[g] for g in dev_root])

rows, draws, fits = [], {}, []
for p, (a, b) in enumerate(FROZEN_PAIRS):
    trm = [n for n in FROZEN_NAMES if n not in (a, b)]
    Ftr = {}
    for m in trm:
        Ftr[m], used = feats(ML4[m], {a, b, m, PARTNER4[m]}); assert not used & {a, b, m, PARTNER4[m]}
    Fte = {}
    for m in (a, b):
        Fte[m], used = feats(ML4[m], {a, b}); assert not used & {a, b}
    for f in range(2):
        tr, te = np.flatnonzero(TFOLD != f), np.flatnonzero(TFOLD == f)
        Xtr = {m: np.column_stack([Ftr[m][c][tr] for c in COLS]) for m in trm}
        sc = StandardScaler().fit(np.vstack(list(Xtr.values())))                      # training rows only
        Zs = [sc.transform(Xtr[m]) for m in trm]; ys = [MEMD[m][tr].astype(int) for m in trm]
        a0 = alpha0(Zs, ys); w0 = np.array([a0, 0.0, 0.0])
        models = {"LR": w0}
        for lam in LAMBDAS:
            w, nr = tail_fit(Zs, ys, lam, w0); models[f"tail λ={lam:g}"] = w
            fits.append({"held_out": f"{a}/{b}", "fold": f, "λ": lam, "rounds": nr, **dict(zip(["w_LR", "w_B_ext", "w_RMIA_a"], w))})
        ordc = LogisticRegression(C=C_ORD, max_iter=5000).fit(np.vstack(Zs), np.concatenate(ys))
        for m in (a, b):
            y = MEMD[m][te].astype(int)
            Z = sc.transform(np.column_stack([Fte[m][c][te] for c in COLS]))
            S = {k: Z @ w for k, w in models.items()} | {"ordinary logistic": ordc.decision_function(Z)}
            assert np.isclose(tpr_strict(y, S["LR"]), tpr_strict(y, Fte[m]["LR"][te]))
            pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
            rng = np.random.default_rng(1000 + 10 * FROZEN_NAMES.index(m) + f)
            bidx = [np.concatenate([rng.choice(pos, pos.size), rng.choice(neg, neg.size)]) for _ in range(1000)]
            for k, v in S.items():
                draws[(m, f, k)] = np.array([strict_fast(v[ix], y[ix]) for ix in bidx])
                rows.append({"target": m, "aux": "fresh" if m in FRESH else "pooled", "fold": f, "method": k,
                             "TPR": tpr_strict(y, v), "AUC": roc_auc_score(y, v)})
cv = pd.DataFrame(rows)
METHODS = list(cv.method.unique())
summ = []
for m in FROZEN_NAMES:
    for k in METHODS:
        sub = cv[(cv.target == m) & (cv.method == k)]
        d = np.mean([draws[(m, f, k)] - draws[(m, f, "LR")] for f in range(2)], axis=0)
        lo, hi = np.percentile(d, [2.5, 97.5])
        summ.append({"target": m, "aux": "fresh" if m in FRESH else "pooled", "method": k, "TPR": sub.TPR.mean(),
                     "AUC": sub.AUC.mean(), "Δ vs LR": sub.TPR.mean() - cv[(cv.target == m) & (cv.method == "LR")].TPR.mean(),
                     "CI vs LR": f"[{lo:+.3f}, {hi:+.3f}]"})
summ = pd.DataFrame(summ)
ov = pd.DataFrame([{"method": k,
                    "TPR fresh": summ[(summ.method == k) & (summ.aux == "fresh")].TPR.mean(),
                    "TPR pooled": summ[(summ.method == k) & (summ.aux == "pooled")].TPR.mean(),
                    "Δ fresh": summ[(summ.method == k) & (summ.aux == "fresh")]["Δ vs LR"].mean(),
                    "Δ pooled": summ[(summ.method == k) & (summ.aux == "pooled")]["Δ vs LR"].mean()} for k in METHODS])
ov["TPR balanced"] = 0.5 * (ov["TPR fresh"] + ov["TPR pooled"]); ov["Δ balanced"] = 0.5 * (ov["Δ fresh"] + ov["Δ pooled"])
pd.set_option("display.width", 220)
print("Held-out targets (pair excluded from rows and references; unseen text groups):")
print(ov.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nFitted weights (standardized) across the 8 folds:")
print(pd.DataFrame(fits).groupby("λ")[["w_LR", "w_B_ext", "w_RMIA_a", "rounds"]].agg(["mean", "min", "max"])
      .to_string(float_format=lambda x: f"{x:+.3f}"))
cand = ov[ov.method.str.startswith("tail")].copy()
cand["λ"] = cand.method.str.extract(r"λ=([\d.]+)").astype(float)
LAM_STAR = float(cand.sort_values(["TPR balanced", "λ"], ascending=[False, False]).iloc[0]["λ"])
print(f"\nSelected λ (balanced held-out TPR; tie → larger λ): {LAM_STAR:g}")
print("\nPer target, selected λ vs LR and ordinary logistic:")
print(summ[summ.method.isin(["LR", "ordinary logistic", f"tail λ={LAM_STAR:g}"])]
      .to_string(index=False, float_format=lambda x: f"{x:.4f}"))
cv.to_csv(TAIL / "cv_rows.csv", index=False); summ.to_csv(TAIL / "cv_summary.csv", index=False)
ov.to_csv(TAIL / "cv_overview.csv", index=False); pd.DataFrame(fits).to_csv(TAIL / "cv_weights.csv", index=False)

Held-out targets (pair excluded from rows and references; unseen text groups):
           method  TPR fresh  TPR pooled  Δ fresh  Δ pooled  TPR balanced  Δ balanced
               LR     0.7442      0.8239   0.0000    0.0000        0.7841      0.0000
      tail λ=0.01     0.7411      0.8243  -0.0031    0.0003        0.7827     -0.0014
       tail λ=0.1     0.7452      0.8239   0.0010   -0.0000        0.7845      0.0005
         tail λ=1     0.7452      0.8239   0.0010    0.0000        0.7846      0.0005
        tail λ=10     0.7442      0.8239   0.0000    0.0000        0.7841      0.0000
ordinary logistic     0.7216      0.8190  -0.0226   -0.0050        0.7703     -0.0138

Fitted weights (standardized) across the 8 folds:
        w_LR               w_B_ext               w_RMIA_a               rounds        
        mean    min    max    mean    min    max     mean    min    max   mean min max
λ                                                                                     
0.01  +

In [32]:
# TAIL-2 — Fit the selected λ on 8 mocks × all dev texts (shadow labels), freeze, evaluate once on p5/p6.
FULL = {}
for m in FROZEN_NAMES:
    FULL[m], used = feats(ML4[m], {m, PARTNER4[m]}); assert not used & {m, PARTNER4[m]}
Xf = {m: np.column_stack([FULL[m][c] for c in COLS]) for m in FROZEN_NAMES}
scF = StandardScaler().fit(np.vstack(list(Xf.values())))
ZsF = [scF.transform(Xf[m]) for m in FROZEN_NAMES]; ysF = [MEMD[m].astype(int) for m in FROZEN_NAMES]
a0F = alpha0(ZsF, ysF); w0F = np.array([a0F, 0.0, 0.0])
wF, nrF = tail_fit(ZsF, ysF, LAM_STAR, w0F)
frozen_tail = {"feature_order": COLS, "scaler_mean": scF.mean_.tolist(), "scaler_scale": scF.scale_.tolist(),
               "w": wF.tolist(), "w0": w0F.tolist(), "lambda": LAM_STAR, "frac_hard_negatives": FRAC,
               "rounds_used": nrF, "training": "8 frozen mocks (refs exclude target+partner) × 2,000 dev texts, shadow labels",
               "selection": "balanced held-out strict TPR over λ ∈ {0.01, 0.1, 1, 10}; tie → larger λ"}
json.dump(frozen_tail, open(TAIL / "frozen_tail_model.json", "w"), indent=1)
print("Frozen tail model weights (standardized):", dict(zip(COLS, np.round(wF, 4))), "| w0:", np.round(w0F, 4))
tail_score = lambda F: scF.transform(np.column_stack([F[c] for c in COLS])) @ wF

rows = []
dd = {"tail vs LR": [], "tail vs ordinary": [], "ordinary vs LR": []}
for t in TARGETS:
    y = MEM[t]["train"].astype(int)
    F, _ = feats(mean_loss(t))                                       # all 8 frozen pairs + 4 fixed (= real-dev setup)
    S = {"LR": F["LR"], "ordinary logistic (frozen E-3)": apply_frozen("LR+B_ext+RMIA_a", F), "tail-rank (frozen)": tail_score(F)}
    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
    rng = np.random.default_rng(700 + TARGETS.index(t))
    bidx = [np.concatenate([rng.choice(pos, pos.size), rng.choice(neg, neg.size)]) for _ in range(2000)]
    bt = {k: np.array([strict_fast(v[ix], y[ix]) for ix in bidx]) for k, v in S.items()}
    for (lab, a_, b_) in (("tail vs LR", "tail-rank (frozen)", "LR"), ("tail vs ordinary", "tail-rank (frozen)", "ordinary logistic (frozen E-3)"),
                          ("ordinary vs LR", "ordinary logistic (frozen E-3)", "LR")):
        dd[lab].append(bt[a_] - bt[b_])
    for k, v in S.items():
        d = bt[k] - bt["LR"]; lo, hi = np.percentile(d, [2.5, 97.5])
        rows.append({"target": t, "method": k, "TPR": tpr_strict(y, v), "AUC": roc_auc_score(y, v),
                     "Δ vs LR": tpr_strict(y, v) - tpr_strict(y, S["LR"]), "95% CI": f"[{lo:+.3f}, {hi:+.3f}]"})
pt = pd.DataFrame(rows)
print("\np5/p6 (exploratory; these models informed this experiment):")
print(pt.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
ms = pt.groupby("method")[["TPR", "AUC", "Δ vs LR"]].mean()
ms["targets Δ>0"] = pt.groupby("method")["Δ vs LR"].apply(lambda s: int((s > 0).sum()))
print("\nMean across the four targets:"); print(ms.to_string(float_format=lambda x: f"{x:.4f}"))
for lab, ds in dd.items():
    dm = np.mean(ds, axis=0); lo, hi = np.percentile(dm, [2.5, 97.5])
    print(f"mean Δ {lab}: {dm.mean():+.4f}  CI [{lo:+.3f}, {hi:+.3f}]")
pt.to_csv(TAIL / "p56_eval.csv", index=False)
json.dump({"p56_status": "exploratory (informed this experiment)", "real_dev_scored": False,
           "validation_submission": False}, open(TAIL / "notes.json", "w"), indent=2)
print("Saved to", TAIL)

Frozen tail model weights (standardized): {'LR': np.float64(3.9916), 'B_ext': np.float64(0.0032), 'RMIA_a': np.float64(0.0013)} | w0: [3.9916 0.     0.    ]

p5/p6 (exploratory; these models informed this experiment):
target                         method    TPR    AUC  Δ vs LR           95% CI
   p5A                             LR 0.7467 0.9794   0.0000 [+0.000, +0.000]
   p5A ordinary logistic (frozen E-3) 0.7549 0.9784   0.0082 [-0.053, +0.045]
   p5A             tail-rank (frozen) 0.7467 0.9794   0.0000 [+0.000, +0.000]
   p5B                             LR 0.7287 0.9778   0.0000 [+0.000, +0.000]
   p5B ordinary logistic (frozen E-3) 0.7130 0.9766  -0.0157 [-0.239, +0.056]
   p5B             tail-rank (frozen) 0.7287 0.9778   0.0000 [-0.003, +0.000]
   p6A                             LR 0.8008 0.9814   0.0000 [+0.000, +0.000]
   p6A ordinary logistic (frozen E-3) 0.7988 0.9799  -0.0020 [-0.078, +0.029]
   p6A             tail-rank (frozen) 0.8008 0.9814   0.0000 [-0.001, +0.000]
  

In [35]:
# RC-2 — ReCaLL kill test: held-out pairs (rows AND references) × text groups; tail-rank combiner. No new inference.
RC = DRIVE / "cache" / "recall"
FIXR = ["shadow1001", "shadow1002", "val_target", "final_target"]
RC_MODELS = ["dev_target"] + FROZEN_NAMES + TARGETS + FIXR
for n in TARGETS:
    MEM[n] = MEM.get(n) or membership([r["text"] for r in read_json(DRIVE / "data" / f"shadow_{n}" / "train_finetune.json")])
RCD, shas = {}, set()
for m in RC_MODELS:
    meta = json.load(open(RC / f"dev_{m}.json")); z = np.load(RC / f"dev_{m}.npz")
    assert meta["status"] == "done" and z["dev_order_sha"].item() == DEV_ORDER_SHA, m
    shas.add(meta["prefix_sha"])
    u, c = z["u"], z["c"]
    assert u.shape == (2000,) and c.shape[1] == 2000 and np.isfinite(u).all() and np.isfinite(c).all()
    RCD[m] = {"d": (c - u[None]).mean(0), "c": c.mean(0), "T": z["T"].astype(float)}
assert len(shas) == 1, "models were extracted with different prefixes"
TP = RCD["dev_target"]["T"]
assert all(np.array_equal(RCD[m]["T"], TP) for m in RC_MODELS)

def rc_feats(mname, excl=()):
    F, used = feats(DEV_MT if mname == "dev_target" else mean_loss(mname), excl)
    names = [n for n in FROZEN_NAMES if n not in set(excl)]
    Pin = np.vstack([np.stack([MEM[n]["train"] for n in names]), np.zeros((4, 2000), bool)])
    for key, st in (("cal_d", "d"), ("LR_c", "c")):
        P = np.vstack([np.stack([RCD[n][st] for n in names]), np.stack([RCD[f][st] for f in FIXR])])
        F[key] = lr_frozen(RCD[mname][st], TP, P, Pin)                  # frozen LR4 formula; direction learned from IN/OUT
        assert np.isfinite(F[key]).all()
    return F, used
assert np.allclose(rc_feats("dev_target")[0]["LR"], LR4_DEV, rtol=0, atol=1e-6)

RC_SETS = {"LR": ["LR"], "LR_c alone": ["LR_c"], "cal_d alone [diag]": ["cal_d"],
           "LR+cal_d": ["LR", "cal_d"], "LR+LR_c+cal_d": ["LR", "LR_c", "cal_d"]}
RC_LAMS = (0.01, 0.1, 1.0)
def make_scorers(Ftr, trm, tr):
    S = {}
    for s, cols in RC_SETS.items():
        if len(cols) == 1:
            S[s] = (lambda F, idx, c=cols[0]: F[c][idx]); continue
        Xtr = {m: np.column_stack([Ftr[m][c][tr] for c in cols]) for m in trm}
        sc = StandardScaler().fit(np.vstack(list(Xtr.values())))
        Zs = [sc.transform(Xtr[m]) for m in trm]; ys = [MEMD[m][tr].astype(int) for m in trm]
        w0 = np.zeros(len(cols)); w0[0] = alpha0(Zs, ys)
        for lam in RC_LAMS:
            w, _ = tail_fit(Zs, ys, lam, w0)
            S[f"{s} | λ={lam:g}"] = (lambda F, idx, sc=sc, w=w, cols=cols: sc.transform(np.column_stack([F[c][idx] for c in cols])) @ w)
    return S

rows, draws = [], {}
for p, (a, b) in enumerate(FROZEN_PAIRS):
    trm = [n for n in FROZEN_NAMES if n not in (a, b)]
    Ftr = {}
    for m in trm:
        Ftr[m], used = rc_feats(m, {a, b, m, PARTNER4[m]}); assert not used & {a, b, m, PARTNER4[m]}
    Fte = {}
    for m in (a, b):
        Fte[m], used = rc_feats(m, {a, b}); assert not used & {a, b}
    for f in range(2):
        tr, te = np.flatnonzero(TFOLD != f), np.flatnonzero(TFOLD == f)
        S = make_scorers(Ftr, trm, tr)
        for m in (a, b):
            y = MEMD[m][te].astype(int)
            pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)
            rng = np.random.default_rng(1000 + 10 * FROZEN_NAMES.index(m) + f)
            bidx = [np.concatenate([rng.choice(pos, pos.size), rng.choice(neg, neg.size)]) for _ in range(1000)]
            for k, fn in S.items():
                v = fn(Fte[m], te)
                draws[(m, f, k)] = np.array([strict_fast(v[ix], y[ix]) for ix in bidx])
                rows.append({"target": m, "aux": "fresh" if m in FRESH else "pooled", "fold": f, "method": k,
                             "TPR": tpr_strict(y, v), "AUC": roc_auc_score(y, v)})
cv = pd.DataFrame(rows); METHODS = list(cv.method.unique())
summ = []
for m in FROZEN_NAMES:
    for k in METHODS:
        sub = cv[(cv.target == m) & (cv.method == k)]
        d = np.mean([draws[(m, f, k)] - draws[(m, f, "LR")] for f in range(2)], axis=0); lo, hi = np.percentile(d, [2.5, 97.5])
        summ.append({"target": m, "aux": "fresh" if m in FRESH else "pooled", "method": k, "TPR": sub.TPR.mean(), "AUC": sub.AUC.mean(),
                     "Δ vs LR": sub.TPR.mean() - cv[(cv.target == m) & (cv.method == "LR")].TPR.mean(), "CI vs LR": f"[{lo:+.3f}, {hi:+.3f}]"})
summ = pd.DataFrame(summ)
ov = pd.DataFrame([{"method": k, **{f"{q} {g}": summ[(summ.method == k) & (summ.aux == g)][col].mean()
                                     for g in ("fresh", "pooled") for q, col in (("TPR", "TPR"), ("AUC", "AUC"), ("Δ", "Δ vs LR"))}}
                   for k in METHODS])
ov["Δ balanced"] = 0.5 * (ov["Δ fresh"] + ov["Δ pooled"]); ov["TPR balanced"] = 0.5 * (ov["TPR fresh"] + ov["TPR pooled"])
pd.set_option("display.width", 240)
print("Held-out targets:"); print(ov.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
cand = ov[(ov.method != "LR") & ~ov.method.str.contains("diag")]
best = cand.sort_values(["TPR balanced"], ascending=False).iloc[0]
PASS = bool(best["Δ balanced"] > 0 and best["Δ fresh"] >= 0)
print(f"\nBest candidate: {best.method} | Δ balanced {best['Δ balanced']:+.4f} | Δ fresh {best['Δ fresh']:+.4f} → "
      f"{'PASS' if PASS else 'FAIL (drop ReCaLL)'}")
print("\nPer target, best candidate vs LR:")
print(summ[summ.method.isin(["LR", best.method, "cal_d alone [diag]"])].to_string(index=False, float_format=lambda x: f"{x:.4f}"))
OUTR = RC / "kill_test"; OUTR.mkdir(exist_ok=True)
cv.to_csv(OUTR / "cv_rows.csv", index=False); summ.to_csv(OUTR / "cv_summary.csv", index=False); ov.to_csv(OUTR / "cv_overview.csv", index=False)

if PASS:
    sname = best.method.split(" | ")[0]; cols = RC_SETS[sname]
    lam = float(best.method.split("λ=")[1]) if "λ=" in best.method else None
    FULLF = {m: rc_feats(m, {m, PARTNER4[m]})[0] for m in FROZEN_NAMES}
    if lam is None:
        score_fn = lambda F: F[cols[0]]; frozen = {"set": sname}
    else:
        Xf = {m: np.column_stack([FULLF[m][c] for c in cols]) for m in FROZEN_NAMES}
        scF = StandardScaler().fit(np.vstack(list(Xf.values())))
        ZsF = [scF.transform(Xf[m]) for m in FROZEN_NAMES]; ysF = [MEMD[m].astype(int) for m in FROZEN_NAMES]
        w0 = np.zeros(len(cols)); w0[0] = alpha0(ZsF, ysF); wF, _ = tail_fit(ZsF, ysF, lam, w0)
        score_fn = lambda F: scF.transform(np.column_stack([F[c] for c in cols])) @ wF
        frozen = {"set": sname, "cols": cols, "lambda": lam, "w": wF.tolist(), "scaler_mean": scF.mean_.tolist(), "scaler_scale": scF.scale_.tolist()}
    json.dump(frozen, open(OUTR / "frozen_model.json", "w"), indent=1)
    er = []
    for t in TARGETS + ["dev_target"]:
        y = (LABELS if t == "dev_target" else MEM[t]["train"].astype(int))
        F_ = rc_feats(t)[0]; v, base_ = score_fn(F_), F_["LR"]
        bix = boot if t == "dev_target" else None
        if bix is None:
            pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0); rng = np.random.default_rng(900 + TARGETS.index(t))
            bix = [np.concatenate([rng.choice(pos, pos.size), rng.choice(neg, neg.size)]) for _ in range(2000)]
        d = np.array([strict_fast(v[ix], y[ix]) - strict_fast(base_[ix], y[ix]) for ix in bix]); lo, hi = np.percentile(d, [2.5, 97.5])
        er.append({"target": t + (" (real, exploratory)" if t == "dev_target" else " (exploratory)"), "LR TPR": tpr_strict(y, base_),
                   "ReCaLL model TPR": tpr_strict(y, v), "Δ": tpr_strict(y, v) - tpr_strict(y, base_), "CI": f"[{lo:+.3f}, {hi:+.3f}]",
                   "LR AUC": roc_auc_score(y, base_), "model AUC": roc_auc_score(y, v)})
    er = pd.DataFrame(er); print("\nFrozen model on p5/p6 and real dev:"); print(er.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    er.to_csv(OUTR / "p56_dev_eval.csv", index=False)
print("Saved to", OUTR)

Held-out targets:
                method  TPR fresh  AUC fresh  Δ fresh  TPR pooled  AUC pooled  Δ pooled  Δ balanced  TPR balanced
                    LR     0.7442     0.9763   0.0000      0.8239      0.9848    0.0000      0.0000        0.7841
            LR_c alone     0.4809     0.9349  -0.2633      0.5964      0.9569   -0.2275     -0.2454        0.5387
    cal_d alone [diag]     0.0580     0.6947  -0.6862      0.0754      0.7181   -0.7486     -0.7174        0.0667
     LR+cal_d | λ=0.01     0.7382     0.9761  -0.0060      0.8225      0.9840   -0.0014     -0.0037        0.7804
      LR+cal_d | λ=0.1     0.7462     0.9764   0.0020      0.8236      0.9848   -0.0003      0.0008        0.7849
        LR+cal_d | λ=1     0.7457     0.9764   0.0015      0.8233      0.9848   -0.0007      0.0004        0.7845
LR+LR_c+cal_d | λ=0.01     0.7437     0.9765  -0.0005      0.8203      0.9842   -0.0037     -0.0021        0.7820
 LR+LR_c+cal_d | λ=0.1     0.7467     0.9764   0.0025      0.8226     

In [36]:
# RC-4 — Frozen LR4 + ReCaLL (RC-2 model) on val and final, with LR4's exact references and masks. Builds ZIPs only.
FRZ = json.load(open(RC / "kill_test" / "frozen_model.json"))
assert FRZ["cols"] == ["LR", "cal_d"], FRZ
W, MU, SD = np.array(FRZ["w"]), np.array(FRZ["scaler_mean"]), np.array(FRZ["scaler_scale"])
FZR = DRIVE / "frozen_LR4_recall"; FZR.mkdir(exist_ok=True)
ORDER_SHA = {ph: hashlib.sha256("\x1e".join(TEXTS[ph]).encode()).hexdigest() for ph in ("val", "final")}
def load_d(ph, m):
    z = np.load(RC / f"{ph}_{m}.npz"); assert z["order_sha"].item() == ORDER_SHA[ph], (ph, m)
    return (z["c"] - z["u"][None]).mean(0), z["T"].astype(float)

SPEC = {"val":   {"target": "val_target",
                  "fixed": [("shadow1001", None), ("shadow1002", None), ("final_target", None), ("dev_target", in_tf["val"])]},
        "final": {"target": "final_target",
                  "fixed": [("shadow1001", mem_s["1001"]["final"]), ("shadow1002", mem_s["1002"]["final"]),
                            ("dev_target", in_tf["final"]), ("val_target", None)]}}
out = {}
for ph, sp in SPEC.items():
    d_t, T = load_d(ph, sp["target"])
    rows = [load_d(ph, n)[0] for n in NAMES4]
    for n, mask in sp["fixed"]:
        d_n = load_d(ph, n)[0]
        rows.append(np.where(mask, np.nan, d_n) if mask is not None else d_n)
    Pd = np.stack(rows)
    Pin = np.vstack([np.stack([MEM[n][ph] for n in NAMES4]), np.zeros((4, 2000), bool)])
    cal_d = lr_frozen(d_t, T, Pd, Pin)
    LR = score(ph, NAMES4)[0]
    saved = np.load(FZ4 / f"{ph}_scores.npz")[f"{ph}_LR"]
    assert np.abs(LR - saved).max() < 1e-4, f"{ph}: LR4 not reproduced"
    s = ((np.column_stack([LR, cal_d]) - MU) / SD) @ W
    assert s.shape == (2000,) and np.isfinite(s).all()
    t4, ts = set(np.argsort(-saved)[:750]), set(np.argsort(-s)[:750])
    print(f"{ph}: NaN-masked fixed refs {int(np.isnan(Pd).sum())} | corr(new, LR4) {np.corrcoef(s, saved)[0, 1]:.5f} | "
          f"Spearman {spearmanr(s, saved)[0]:.5f} | top-750 overlap {len(t4 & ts)}/750")
    out[ph] = s
    pred = ROOT / "data" / ph / "prediction_LR4_recall.csv"; zp = ROOT / f"project_phase2_{ph}_submission_LR4_recall.zip"
    np.savetxt(pred, s, fmt="%.17g")
    for cmd in ([sys.executable, str(ROOT / "scripts" / "make_submission.py"), "--phase", ph, "--prediction", str(pred), "--output", str(zp)],
                [sys.executable, str(ROOT / "scripts" / "validate_submission.py"), str(zp), "--phase", ph]):
        r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True); print(r.stdout.strip(), r.stderr.strip()); assert r.returncode == 0
    with zipfile.ZipFile(zp) as zf:
        assert np.allclose(np.loadtxt(io.BytesIO(zf.read(f"{ph}/prediction.csv"))), s, rtol=0, atol=1e-9)
    shutil.copy(zp, FZR); shutil.copy(zp, DRIVE)
    np.savez(FZR / f"{ph}_scores.npz", score=s, LR=LR, cal_d=cal_d)
json.dump({"model": "frozen RC-2: tail-rank LR+cal_d, λ=0.1", "w": FRZ["w"], "refs": "LR4's exact val/final references and masks",
           "decision_rule": "replace LR4 for final only if val strict TPR ≥ 0.790; otherwise keep LR4"},
          open(FZR / "settings.json", "w"), indent=2)
print("Built: project_phase2_val_submission_LR4_recall.zip (and final). Upload the val ZIP once.")

val: NaN-masked fixed refs 39 | corr(new, LR4) 0.99997 | Spearman 0.99995 | top-750 overlap 749/750
Created /content/Project-phase-2/project_phase2_val_submission_LR4_recall.zip with 2000 scores 
Submission validation: PASS (2000 scores)
Score range: [-13.9068, 27.5516] 
final: NaN-masked fixed refs 83 | corr(new, LR4) 0.99996 | Spearman 0.99994 | top-750 overlap 747/750
Created /content/Project-phase-2/project_phase2_final_submission_LR4_recall.zip with 2000 scores 
Submission validation: PASS (2000 scores)
Score range: [-15.1241, 31.9359] 
Built: project_phase2_val_submission_LR4_recall.zip (and final). Upload the val ZIP once.
